# Car Price Prediction — Model Training

In this notebook, we will train and evaluate machine learning models to predict vehicle prices.

We will start by splitting the cleaned dataset into training and testing sets, encoding categorical features, and establishing a baseline.

We will then train regression models and compare their predictive performance.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Loading the Cleaned Dataset

We load the processed dataset generated during data preprocessing.

This dataset contains the nine selected features and the target variable (`price`), with missing values and identified inconsistencies already handled.

In [3]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

file_path = project_root / "data" / "processed" / "cars_clean.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (108273, 10)


,mileage_numeric,vehicle_age,power_hp,make,model,fuel_category,transmission,body_type,country_code,price
0,10500.0,1.0,83.0,Suzuki,Swift,Gasoline,Manual,Compact,NL,22950.0
1,55609.0,5.0,360.0,BMW,X3 M,Gasoline,Automatic,Off-Road/Pick-up,DE,46000.0
2,18200.0,7.0,500.0,Porsche,991,Gasoline,Manual,Coupe,BE,179900.0
3,10.0,0.0,204.0,Audi,A6,Electric/Diesel,Automatic,Station wagon,IT,70400.0
4,74602.0,6.0,330.0,Porsche,Panamera,Electric/Gasoline,Automatic,Station wagon,NL,66999.0


## Preparing Features and Target

We separate the selected vehicle characteristics (X) from the target variable (y), which represents the vehicle price.

The model will use nine features to learn how different vehicle characteristics influence the advertised price.

In [4]:
numerical_features = ["mileage_numeric", "vehicle_age", "power_hp"]

categorical_features = ["make", "model", "fuel_category", "transmission", "body_type", "country_code"]

features = numerical_features + categorical_features

X = df[features].copy()
y = df["price"].copy()

print("Features:", X.shape)
print("Target:", y.shape)

Features: (108273, 9)
Target: (108273,)


## Train-Test Split

We split the dataset into 80% training data and 20% testing data.

The training set will be used to train our models, while the testing set will be used to evaluate predictions on unseen vehicles.

A fixed random state ensures reproducible results.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 86618
Testing records: 21655


## Categorical Encoding

We use One-Hot Encoding to convert categorical features into numerical representations without introducing an artificial order between categories.

Numerical features remain unchanged. The encoder is fitted only on the training data to prevent data leakage.

Other encoding techniques may be explored during future model optimization.

In [6]:
preprocessor = ColumnTransformer(transformers=[
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numerical", "passthrough", numerical_features)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

Training shape: (86618, 662)
Testing shape: (21655, 662)


## Baseline Model

Before training machine learning algorithms, we establish a baseline using the median vehicle price from the training set.

The baseline predicts the same price for every vehicle, regardless of its characteristics.

We will evaluate its performance using Mean Absolute Error (MAE), which measures the average absolute difference between actual and predicted prices.

This result will serve as a reference for comparing more advanced regression models.

In [7]:
# Create and train the baseline model
baseline_model = DummyRegressor(strategy="median")
baseline_model.fit(X_train_encoded, y_train)

# Predict car prices
baseline_predictions = baseline_model.predict(X_test_encoded)

# Calculate the prediction error
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Training median price: €{y_train.median():,.2f}")
print(f"Baseline MAE: €{baseline_mae:,.2f}")

# Compare actual and predicted prices
comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": baseline_predictions
})

comparison.head()

Training median price: €39,485.00
Baseline MAE: €34,978.86


,Actual Price,Predicted Price
0,44890.0,39485.0
1,1850.0,39485.0
2,97990.0,39485.0
3,150900.0,39485.0
4,1199.0,39485.0


### Baseline Results

The baseline predicts a constant price of €39,485 for every vehicle.

It achieved a Mean Absolute Error (MAE) of €34,978.86 on the test set.

This relatively high error reflects the limitations of predicting a single value for vehicles with widely varying prices.

We will use this result as a reference to evaluate whether more advanced regression models can produce better predictions.

## Linear Regression

We will train a Linear Regression model to predict vehicle prices using the selected numerical and categorical features.

Unlike the baseline, Linear Regression learns relationships between vehicle characteristics and their prices.

We will evaluate its performance using Mean Absolute Error (MAE) and compare the result with our baseline.

In [8]:
# Create and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train)

# Predict car prices
linear_predictions = linear_model.predict(X_test_encoded)

# Calculate the prediction error
linear_mae = mean_absolute_error(y_test, linear_predictions)

print(f"Baseline MAE: €{baseline_mae:,.2f}")
print(f"Linear Regression MAE: €{linear_mae:,.2f}")

# Compare actual and predicted prices
comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": linear_predictions
})

comparison.head()

Baseline MAE: €34,978.86
Linear Regression MAE: €20,882.85


,Actual Price,Predicted Price
0,44890.0,47210.684777
1,1850.0,-3175.488160
2,97990.0,76057.288013
3,150900.0,142783.477976
4,1199.0,-9783.402354


### Linear Regression Results

The Linear Regression model achieved a MAE of €20,882.85, reducing the baseline error by approximately 40.3%.

Although the model significantly improved prediction accuracy, some predictions were negative, which is not realistic for vehicle prices.

We will investigate how frequently negative predictions occur before deciding whether additional improvements are necessary.

In [9]:
negative_predictions = linear_predictions < 0

print("Total predictions:", len(linear_predictions))
print("Negative predictions:", negative_predictions.sum())
print("Negative predictions (%):", round(negative_predictions.mean() * 100, 2))

Total predictions: 21655
Negative predictions: 1690
Negative predictions (%): 7.8


**Negative Predictions — Findings**

Of the 21,655 test predictions, 1,690 (7.8%) were negative.

Although Linear Regression reduced the baseline MAE by approximately 40.3%, these negative predictions reveal an important limitation: the model does not guarantee realistic, positive vehicle prices.

We will keep these results unchanged and compare them with a more flexible regression algorithm.

## 8. Random Forest Regression

We will train a Random Forest Regressor to capture more complex relationships between vehicle characteristics and prices.

Unlike Linear Regression, Random Forest combines predictions from multiple decision trees and can model non-linear patterns.

We will evaluate its performance using MAE and compare it with the Baseline and Linear Regression models.

In [10]:
# Create and train the Random Forest model
forest_model = RandomForestRegressor(n_estimators=50, max_depth=18, min_samples_leaf=3, random_state=42, n_jobs=2)
forest_model.fit(X_train_encoded, y_train)

# Predict car prices
forest_predictions = forest_model.predict(X_test_encoded)

# Calculate the prediction error
forest_mae = mean_absolute_error(y_test, forest_predictions)

print(f"Baseline MAE: €{baseline_mae:,.2f}")
print(f"Linear Regression MAE: €{linear_mae:,.2f}")
print(f"Random Forest MAE: €{forest_mae:,.2f}")

# Compare actual and predicted prices
forest_comparison = pd.DataFrame({
    "Actual Price": y_test.to_numpy(),
    "Predicted Price": forest_predictions
})

forest_comparison.head()

Baseline MAE: €34,978.86
Linear Regression MAE: €20,882.85
Random Forest MAE: €6,359.84


,Actual Price,Predicted Price
0,44890.0,41094.916011
1,1850.0,3724.120633
2,97990.0,92543.789256
3,150900.0,148492.215239
4,1199.0,2500.669122


### Random Forest Results

The Random Forest model achieved a MAE of €6,359.84, reducing the error by approximately 69.5% compared to Linear Regression and 81.8% compared to the Baseline.

The model produced more accurate predictions and avoided the negative price predictions observed with Linear Regression.

These results suggest that non-linear relationships are important for vehicle price prediction.

However, we still need to check whether the model is overfitting the training data.

### Checking for Overfitting

We will compare the Random Forest's performance on the training and testing datasets.

A significantly lower training error may indicate that the model has learned patterns specific to the training data rather than generalizing well to unseen vehicles.

This comparison will help us assess whether the model needs further adjustments.

In [11]:
# Predict prices for training vehicles
forest_train_predictions = forest_model.predict(X_train_encoded)

# Calculate training error
forest_train_mae = mean_absolute_error(y_train, forest_train_predictions)

# Compare training and testing errors
print(f"Random Forest Training MAE: €{forest_train_mae:,.2f}")
print(f"Random Forest Testing MAE: €{forest_mae:,.2f}")

Random Forest Training MAE: €4,915.56
Random Forest Testing MAE: €6,359.84


### Overfitting Analysis — Findings

The Random Forest achieved a Training MAE of €4,915.56 and a Testing MAE of €6,359.84.

The testing error is approximately 29.4% higher than the training error, indicating a moderate performance gap.

Although the model performs better on the training data, the results do not provide clear evidence of severe overfitting.

Further validation would help assess its ability to generalize to unseen vehicles.